## Install google gemini sdk for langchain in uv
- Run command `uv add langchain-google-genai`
- Setup Google gemini API KEY at https://ai.google.dev/gemini-api/docs/api-key
- Add `GOOGLE_API_KEY/GEMINI_API_KEY` in .env FILE. The integration checks for GOOGLE_API_KEY first, then GEMINI_API_KEY as a fallback.


In [1]:
# Load the system variables
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=100,
    timeout=200,
    max_retries=2,
)

messages=[
    ('system','You are a expert in Indian history. Return me 5 facts about any freedom fighter name that I give you.'),
    ('human','Mahatama gandhi')
]

result = model.invoke(messages)

print(result.content)

[{'type': 'text', 'text': 'Here are 5 fascinating facts about Mahatma Gandhi:\n\n1.  **He did not hold any official position in the Indian National Congress:** Despite being the supreme leader of the Indian independence movement and the face of the Congress party, Gandhi never held any formal office within the organization. He remained an ordinary member for most of his life, believing that his influence should come from his moral authority rather than political power.\n2.  **The "Mahatma" title origin:** The', 'extras': {'signature': 'EjQKMgEMOdbHoxz+b7jtW6mZ1zl/4Ed4ZxtrRvbDPbNX2iclenf/3WgGluwoOphXQMV4sjSW'}}]


In [ ]:
from pydantic import BaseModel
import json
from typing import Optional, List


class FreedomFighterResponse(BaseModel):
    name:Optional[str]= None
    facts:List[str]

modelWithStructOutput = model.with_structured_output(
    schema=FreedomFighterResponse.model_json_schema(),
    method="json_schema",
)

result = modelWithStructOutput.invoke(messages)
# print(result)

fighterObject:FreedomFighterResponse = FreedomFighterResponse.model_validate_json(json.dumps(result))

# Save object to a json file in the current directory
with open('./fighterObject.json', 'w') as f:
    json.dump(result, f, indent=4)


In [15]:
from pydantic import BaseModel,Field
import json
from typing import Optional, List


class FreedomFighterResponse(BaseModel):
    name:Optional[str]= Field(description="Freedom fighter name")
    facts:List[str] = Field(description="List of facts about the freedom fighter")

structured_model = model.with_structured_output(FreedomFighterResponse)

result = structured_model.invoke(messages)
print(result.facts[0])



He pioneered the philosophy of Satyagraha, which is resistance to tyranny through mass non-violent civil disobedience.
